# Hệ thống Phân tích Cảm xúc TikTok Song ngữ (Tiếng Việt & Tiếng Anh)

- **Nguồn đầu vào (2 chế độ):** **1. Link TikTok** (Chuẩn bị dữ liệu: tự tải video/ảnh, crawl comment + reply và phân tích nội dung video lưu vào `data/<post_id>/`) hoặc **2. Thư mục đã chuẩn bị sẵn** (`data/<post_id>/` chứa `raw_comments.csv`, video/ảnh & `media_context.json`).
- **Chuẩn bị dữ liệu & Giai đoạn 1 (`gemma4-e2b-opt` + Whisper):** Trích khung hình có mốc thời gian (`768px`, tự động 2-pass cho slideshow dài) + lời thoại Whisper + caption thành JSON Evidence-First (11 trường) và lưu vào `data/<post_id>/media_context.json` để tái sử dụng cho Giai đoạn 2–3 mà không cần chạy lại Gemma 4.
- **Giai đoạn 2 — Tiền xử lý & Lọc nhiễu có ngữ cảnh (`src/preprocessing/text_cleaner.py`):** Nhận diện ngôn ngữ (`vi` / `en`), bảo vệ thực thể video không bị lọc nhầm, lọc nhiễu, tách từ `underthesea` và tính độ liên quan chủ đề (`is_on_topic`, `context_relevance`).
- **Giai đoạn 3 — Lõi Phân tích Cảm xúc (`src/models/classical_ml.py` + `sentiment_engine.py`):** Cho phép chọn giữa **2 chế độ** ngay khi chạy Cell 1 (mặc định `FULL_MODELS = False` chạy **5 đại diện cho 5 họ ML + Smart Meta-Ensemble** cho gọn và nhanh; đặt `FULL_MODELS = True` để xuất đủ 10+1 mô hình):
  - **`1. ML truyền thống` (`"classical"`):** 5 họ Classical ML v3 + `Bilingual Lexicon` (**0% Deep Learning** — chạy rất nhanh).
  - **`2. Như hiện tại` (`"hybrid"`, mặc định):** Classical ML v3 + `Bilingual Lexicon` + Deep Learning (`ViSoBERT`, `PhoBERT`, `Twitter-RoBERTa`, `DistilRoBERTa`).
- **Giai đoạn 4 — AI Đầu ra (`src/models/gemma_auditor.py`):** Notebook sẽ **luôn hỏi tương tác** cả khi chọn Option 1 lẫn Option 2 (`data/<post_id>/`). Khi bật, có thể chọn thẩm định **chỉ các ca khó (`uncertain`)** hoặc **toàn bộ bình luận (`all`)**, kèm kiểm tra kết nối Ollama trước khi chạy; xuất **8 tệp** ra `outputs/<run_name>/` (`report.html`, `final_results.csv`, `detailed_model_outputs.csv`, `filtered_noise.csv`, `model_comparison.csv`, `report.json`, `dashboard_overview.png`, `dashboard_models.png`).

> Lưu ý: Chỉ cần khởi động container Ollama `gemma4-e2b` khi trích xuất mới `media_context.json` ở Giai đoạn 1 hoặc khi bật Giai đoạn 4 (`RUN_AI_AUDIT = True`).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.pipeline import SentimentPipeline, normalize_engine_mode
from src.models.sentiment_engine import CommentAnalyzer
from src.visualization import plot_models_dashboard, plot_overview_dashboard

# === TÙY CHỌN GÁN CỨNG (để None nếu muốn notebook hỏi tương tác khi chạy) ===
INPUT_MODE = None            # None (hỏi trực tiếp) | "url" | "folder"
TIKTOK_URL = ""
INPUT_DIR = None             # Đường dẫn thư mục đã chuẩn bị sẵn (vd: "data/7707163951179549973") cho chế độ "folder"
RUN_MEDIA_ANALYSIS = True    # True = dùng/trích xuất media_context.json; False = bỏ qua ngữ cảnh video
FORCE_REANALYZE_MEDIA = False # True = ép Gemma 4 + Whisper phân tích lại dù đã có media_context.json
PREPARE_ONLY = False         # True = chỉ Chuẩn bị dữ liệu vào data/<post_id>/ rồi dừng
ENGINE_MODE = None           # None (hỏi 1/2) | "1"/"classical" (ML truyền thống) | "2"/"hybrid" (như hiện tại)
FULL_MODELS = False          # False = 5 đại diện cho 5 họ ML + Ensemble (gọn, nhanh) | True = đủ 10+1 mô hình
RETRAIN_ML = False           # True = huấn luyện lại Classical ML v3 (VI/EN) trước khi chạy
RUN_AI_AUDIT = None          # None (hỏi y/N khi chạy) | True (ép bật GĐ 4) | False (ép tắt GĐ 4)
AUDIT_SCOPE = None           # None (hỏi khi bật GĐ 4) | "uncertain" (chỉ ca khó) | "all" (toàn bộ)

# === CHỌN NGUỒN ĐẦU VÀO ===
if INPUT_MODE is None:
    print("Chọn nguồn dữ liệu đầu vào:")
    print("  1. Link TikTok (Chuẩn bị dữ liệu: tải media, crawl comment + reply & lưu vào data/<post_id>/)")
    print("  2. Thư mục đã chuẩn bị sẵn (vd: data/<post_id>)")
    _choice = input("Nhập lựa chọn [1/2, mặc định 1]: ").strip() or "1"
    INPUT_MODE = {"1": "url", "2": "folder"}.get(_choice, "url")

if INPUT_MODE == "url":
    if not TIKTOK_URL:
        TIKTOK_URL = input("Nhập link video/ảnh TikTok: ").strip().strip("\"'")
elif INPUT_MODE == "folder":
    if not INPUT_DIR:
        INPUT_DIR = input("Nhập đường dẫn thư mục (vd: data/7707163951179549973): ").strip().strip("\"'")
else:
    raise ValueError(f"INPUT_MODE không hợp lệ: {INPUT_MODE} (chỉ hỗ trợ 'url' hoặc 'folder').")

# === TÙY CHỌN GIAI ĐOẠN 3 (1 = ML TRUYỀN THỐNG | 2 = HYBRID NHƯ HIỆN TẠI) ===
if not PREPARE_ONLY:
    if ENGINE_MODE is None:
        print("\nChọn chế độ Giai đoạn 3 (Phân tích cảm xúc):")
        print("  1. ML truyền thống (5 họ Classical ML v3 + Lexicon, 0% Deep Learning — nhanh)")
        print("  2. Như hiện tại (Hybrid: Classical ML v3 + Lexicon + Deep Learning VI/EN)")
        _eng_choice = input("Nhập lựa chọn Giai đoạn 3 [1/2, mặc định 2]: ").strip() or "2"
        ENGINE_MODE = normalize_engine_mode(_eng_choice)
    else:
        ENGINE_MODE = normalize_engine_mode(ENGINE_MODE)
else:
    ENGINE_MODE = normalize_engine_mode(ENGINE_MODE)

# === TÙY CHỌN GIAI ĐOẠN 4 (THẨM ĐỊNH BẰNG AI GEMMA 4) ===
if not PREPARE_ONLY:
    if RUN_AI_AUDIT is None:
        _ans = input(
            "\nCó chạy Giai đoạn 4 (dùng AI Gemma 4 thẩm định ca mỉa mai sâu & đối chiếu mô hình) không? [y/N, mặc định N]: "
        ).strip().lower()
        RUN_AI_AUDIT = _ans in {"y", "yes", "c", "co", "có", "1"}

    if RUN_AI_AUDIT:
        if AUDIT_SCOPE is None:
            print("Chọn phạm vi thẩm định Giai đoạn 4:")
            print("  1. Chỉ các ca khó (needs_audit=True — nhanh, tiết kiệm thời gian)")
            print("  2. Toàn bộ bình luận hợp lệ (để tính đầy đủ Macro-F1 & Cohen's Kappa)")
            _scope_choice = input("Nhập lựa chọn phạm vi Giai đoạn 4 [1/2, mặc định 2]: ").strip() or "2"
            AUDIT_SCOPE = "uncertain" if _scope_choice == "1" else "all"
    else:
        AUDIT_SCOPE = AUDIT_SCOPE or "all"
else:
    RUN_AI_AUDIT = False
    AUDIT_SCOPE = "all"

# === CẤU HÌNH HỆ THỐNG ===
ENABLE_AUDIO = True
AUDIT_LIMIT = None             # None = chấm 100% tập đã chọn theo AUDIT_SCOPE
MAX_COMMENTS = 2000            # Số lượng bình luận tối đa cần thu thập (cả comment + reply)
INCLUDE_REPLIES = True         # Thu thập cả bình luận trả lời (reply)
MAX_REPLIES_PER_COMMENT = None # None = lấy toàn bộ reply tới khi chạm MAX_COMMENTS
DEDUPLICATE_STATS = False      # True nếu muốn loại bỏ bình luận copy-paste khỏi tỷ lệ %
DATA_DIR = Path("data")        # Nơi lưu toàn bộ dữ liệu chuẩn bị đầu vào (video, comments, media_context.json)
OUTPUT_DIR = Path("outputs")   # Nơi xuất kết quả cuối cùng sau xử lý

_engine_desc = "1. ML truyền thống (classical)" if ENGINE_MODE == "classical" else "2. Hybrid hiện tại (ML v3 + DL)"
print(
    f"-> Chế độ đầu vào: {INPUT_MODE} | GĐ 3: {_engine_desc} | "
    f"GĐ 4 (AI Audit): {RUN_AI_AUDIT}" + (f" ({AUDIT_SCOPE})" if RUN_AI_AUDIT else "")
)

In [ ]:
# === CHẠY TOÀN BỘ PIPELINE ===
if INPUT_MODE == "url" and not TIKTOK_URL:
    raise ValueError("Vui lòng nhập link TikTok vào biến TIKTOK_URL.")
if INPUT_MODE == "folder" and not INPUT_DIR:
    raise ValueError("Vui lòng nhập đường dẫn thư mục vào biến INPUT_DIR.")

from IPython.display import HTML, display

comment_analyzer = CommentAnalyzer(
    engine_mode=normalize_engine_mode(ENGINE_MODE),
    full_models=bool(FULL_MODELS),
)
if RETRAIN_ML:
    comment_analyzer.classical_suite.load_or_train("vi", force_retrain=True)
    comment_analyzer.classical_suite.load_or_train("en", force_retrain=True)
pipeline = SentimentPipeline(comment_analyzer=comment_analyzer)
report = pipeline.run(
    tiktok_url=TIKTOK_URL if INPUT_MODE == "url" else None,
    input_dir=INPUT_DIR if INPUT_MODE == "folder" else None,
    enable_audio=ENABLE_AUDIO,
    run_media_analysis=RUN_MEDIA_ANALYSIS,
    force_reanalyze_media=FORCE_REANALYZE_MEDIA,
    prepare_only=PREPARE_ONLY,
    run_ai_audit=bool(RUN_AI_AUDIT),
    audit_limit=AUDIT_LIMIT,
    audit_scope=AUDIT_SCOPE or "all",
    engine_mode=ENGINE_MODE,
    full_models=bool(FULL_MODELS),
    max_comments=MAX_COMMENTS,
    include_replies=INCLUDE_REPLIES,
    max_replies_per_comment=MAX_REPLIES_PER_COMMENT,
    deduplicate_stats=DEDUPLICATE_STATS,
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
)

run_name = report["run_name"]
if report.get("prepare_only"):
    print(f"-> Đã chuẩn bị xong dữ liệu tại: {report.get('data_dir')}")
    print(f"-> Tệp media_context.json: {report.get('media_context_path')}")
else:
    print(f"-> Kết quả xử lý xuất tại: {OUTPUT_DIR / run_name} (Tổng thời gian: {report.get('total_duration_sec', 0)}s)")
    if report.get("pipeline_stages"):
        print("\n=== TIẾN TRÌNH 4 GIAI ĐOẠN ===")
        display(pd.DataFrame([
            {"Giai đoạn": f"{s['id']} — {s['title']}", "Trạng thái": s["status"], "Thời gian (s)": s["duration_sec"], "Tóm tắt": s["summary"]}
            for s in report["pipeline_stages"]
        ]))
    html_rep = (report.get("exported_files") or {}).get("report_html")
    if html_rep:
        display(HTML(f'<p><strong>Báo cáo tương tác (mở trình duyệt):</strong> <a href="{Path(html_rep).as_posix()}" target="_blank">{html_rep}</a></p>'))

if report.get("media_analysis"):
    m = report["media_analysis"]
    print("\n=== NGỮ CẢNH VIDEO / ẢNH ===")
    display(pd.DataFrame([
        {"Trường": "Chủ đề chính", "Nội dung": m.get("chu_de_chinh", "")},
        {"Trường": "Mục đích", "Nội dung": m.get("muc_dich_chinh", "")},
        {"Trường": "Sắc thái bài", "Nội dung": m.get("sac_thai_noi_dung", "")},
        {"Trường": "Đối tượng chính", "Nội dung": ", ".join(m.get("doi_tuong_chinh") or [])},
        {"Trường": "Tóm tắt ngắn", "Nội dung": m.get("tom_tat_ngan", "")},
    ]))

In [ ]:
# === XEM KẾT QUẢ TÁCH BIỆT COMMENT SẠCH, CHI TIẾT MÔ HÌNH & COMMENT NHIỄU ===
if report.get("prepare_only"):
    print("Đang ở chế độ PREPARE_ONLY=True (chưa chạy Giai đoạn 2 & 3). Hãy đặt PREPARE_ONLY=False để phân tích cảm xúc.")
else:
    print("=== THỐNG KÊ LỌC NHIỄU & NGÔN NGỮ ===")
    print(json.dumps(report["noise_filter_summary"], ensure_ascii=False, indent=2))
    if report.get("sentiment_summary"):
        print("Phân bố ngôn ngữ hợp lệ:", report["sentiment_summary"]["language_counts"])
        if report["sentiment_summary"].get("on_topic_summary"):
            print("Thống kê nhóm bám sát nội dung video:", json.dumps(report["sentiment_summary"]["on_topic_summary"], ensure_ascii=False, indent=2))

    exported = report.get("exported_files", {})
    if "final_results_csv" in exported:
        df_clean = pd.read_csv(exported["final_results_csv"], encoding="utf-8-sig")
        print(f"\nBảng kết quả dễ đọc (final_results.csv — {len(df_clean.columns)} cột, đã sắp theo lượt thích giảm dần): {len(df_clean)} dòng")
        display(df_clean.head(20))

    if "detailed_model_outputs_csv" in exported:
        df_detail = pd.read_csv(exported["detailed_model_outputs_csv"], encoding="utf-8-sig")
        print(f"\nBảng chi tiết mô hình, ngữ cảnh video & xác suất (detailed_model_outputs.csv — {len(df_detail.columns)} cột): {len(df_detail)} dòng")
        display(df_detail.head(10))

    if "filtered_noise_csv" in exported:
        df_noise = pd.read_csv(exported["filtered_noise_csv"], encoding="utf-8-sig")
        print(f"\nSố comment bị lọc nhiễu (filtered_noise.csv): {len(df_noise)}")
        display(df_noise.head(20))

In [ ]:
# === SO SÁNH 5 HỌ CLASSICAL ML v3 VS DEEP LEARNING VS HYBRID ===
if report.get("prepare_only"):
    print("Đang ở chế độ PREPARE_ONLY=True (bỏ qua so sánh mô hình).")
else:
    bench = (report.get("sentiment_summary") or {}).get("classical_ml_benchmark") or {}
    for lang, b in bench.items():
        print(f"=== BENCHMARK TEST SET ({lang.upper()} v{b.get('schema_version', 1)}) ===")
        if b.get("polarity_models"):
            display(pd.DataFrame.from_dict(b["polarity_models"], orient="index")[["accuracy", "macro_f1", "weighted_f1"]])
        if b.get("emotion_models"):
            display(pd.DataFrame.from_dict(b["emotion_models"], orient="index")[["accuracy", "macro_f1", "weighted_f1"]])

    comp = report.get("model_comparison", {})
    if comp.get("polarity_models"):
        print("\n=== BẢNG SO SÁNH MÔ HÌNH POLARITY ===")
        df_pol_comp = pd.DataFrame(comp["polarity_models"])
        display(df_pol_comp.dropna(axis=1, how="all"))

    if comp.get("emotion_models"):
        print("\n=== BẢNG SO SÁNH MÔ HÌNH 7 CẢM XÚC ===")
        df_emo_comp = pd.DataFrame(comp["emotion_models"])
        display(df_emo_comp.dropna(axis=1, how="all"))

    audit_status = report.get("ai_audit_status") or {}
    if report.get("ai_audit_verdict"):
        print("\n=== KẾT LUẬN THẨM ĐỊNH ĐẦU RA (GEMMA 4 E2B) ===")
        if audit_status.get("message"):
            print(audit_status["message"])
        print(json.dumps(report["ai_audit_verdict"], ensure_ascii=False, indent=2))
    elif audit_status.get("requested"):
        print(f"\n[Cảnh báo Giai đoạn 4] {audit_status.get('message', 'Không thực hiện được thẩm định AI.')}")
    else:
        print(f"\n[Giai đoạn 4] {audit_status.get('message', 'Đã bỏ qua thẩm định AI theo cấu hình người dùng (RUN_AI_AUDIT=False).')}")

    if report.get("keywords_tfidf"):
        print("\n=== TỪ KHÓA NỔI BẬT (TF-IDF) ===")
        kw_rows = [
            {"Nhóm": grp, "Từ khóa": item["keyword"], "TF-IDF": item["tfidf_score"]}
            for grp in ("overall", "positive", "negative")
            for item in (report["keywords_tfidf"].get(grp) or [])[:6]
        ]
        if kw_rows:
            display(pd.DataFrame(kw_rows))

In [ ]:
# === DASHBOARD TRỰC QUAN HÓA ===
if report.get("prepare_only"):
    print("Đang ở chế độ PREPARE_ONLY=True (không vẽ dashboard).")
else:
    summary = report.get("sentiment_summary") or {}
    if summary.get("total_valid_comments", 0) > 0:
        exported = report.get("exported_files", {})
        ov_path = exported.get("dashboard_overview_png") or (OUTPUT_DIR / report["run_name"] / "dashboard_overview.png")
        mod_path = exported.get("dashboard_models_png") or (OUTPUT_DIR / report["run_name"] / "dashboard_models.png")
        fig1 = plot_overview_dashboard(report, save_path=ov_path)
        plt.show()
        fig2 = plot_models_dashboard(report, save_path=mod_path)
        plt.show()
    else:
        print("Không có bình luận hợp lệ để vẽ dashboard.")